# Frozen S1 Kaggle development execution
Run cells in order with Internet enabled and GPU accelerator selected. This is development-only: confirmation, Gate 6 human rows, and E0/E1/E2 are forbidden. The mandatory canary stops execution on any failure.

In [ ]:
%pip install -q transformers==4.51.3 accelerate==1.6.0 bitsandbytes==0.45.5 sentencepiece==0.2.0 scikit-learn==1.6.1 psutil==6.1.1


In [ ]:
import json, os, pathlib, shutil, subprocess, sys
from kaggle_secrets import UserSecretsClient
FROZEN_SHA = '38f9afb479f738081301172176d3133e4056bd7c'
REMOTE_URL = 'https://github.com/mahmutovichana/MASTER-RAD-PROJEKAT.git'
secret = UserSecretsClient().get_secret('HF_TOKEN')
if not secret: raise RuntimeError('Kaggle Secret HF_TOKEN is unavailable')
os.environ['HF_TOKEN'] = secret
del secret
runtime_candidates = [p.parents[2] for p in pathlib.Path('/kaggle/input').glob('**/experiments/posthoc_stage3_s1/artifact_manifest.json') if (p.parents[0] / 'scripts/package_s1_development_return.py').is_file()]
if len(runtime_candidates) != 1: raise RuntimeError(f'Attach exactly one exported S1 runtime asset dataset; found {len(runtime_candidates)}')
runtime_root = runtime_candidates[0]
print('HF_TOKEN loaded: yes; value suppressed')
print('Runtime assets:', runtime_root)


In [ ]:
repo = pathlib.Path('/kaggle/working/MRS1-frozen')
if repo.exists(): shutil.rmtree(repo)
clone_env = dict(os.environ); clone_env['GIT_LFS_SKIP_SMUDGE'] = '1'
subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout', REMOTE_URL, str(repo)], env=clone_env, check=True)
subprocess.run(['git', 'fetch', '--no-tags', 'origin', FROZEN_SHA], cwd=repo, env=clone_env, check=True)
subprocess.run(['git', 'checkout', '--detach', FROZEN_SHA], cwd=repo, env=clone_env, check=True)
head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=repo, check=True, capture_output=True, text=True).stdout.strip()
if head != FROZEN_SHA: raise RuntimeError(f'Frozen checkout mismatch: {head}')
subprocess.run(['git', 'lfs', 'install', '--local'], cwd=repo, check=True)
lfs_include = ','.join(['reports/final_v2/gate4/primary_sample.jsonl','reports/final_v2/gate4/secondary_stress_sample.jsonl','experiments/consolidated_enriched_training_v2/gold/train.jsonl','experiments/consolidated_enriched_training_v2/gold/validation.jsonl'])
subprocess.run(['git', 'lfs', 'pull', '--include', lfs_include], cwd=repo, check=True)
for rel in lfs_include.split(','):
    if (repo / rel).read_bytes()[:80].startswith(b'version https://git-lfs.github.com/spec'): raise RuntimeError(f'Unmaterialized LFS pointer: {rel}')
print('Frozen remote checkout and required Git LFS objects: PASS', head)


In [ ]:
env = dict(os.environ)
env['PYTHONPATH'] = os.pathsep.join([str(runtime_root), str(repo), env.get('PYTHONPATH','')])
output = pathlib.Path('/kaggle/working/s1-development')
output.mkdir(parents=True, exist_ok=True)
canary_receipt = output / 'canary_receipt.json'
canary = subprocess.run([sys.executable, '-m', 'experiments.posthoc_stage3_s1.scripts.kaggle_development_runner', '--canary', '--receipt', str(canary_receipt)], cwd=runtime_root, env=env)
receipt = json.loads(canary_receipt.read_text())
print(receipt['state'])
if canary.returncode != 0 or receipt.get('state') != 'CANARY_PASS': raise RuntimeError('CANARY_STOP: 200-row development study was not executed')


In [ ]:
subprocess.run([sys.executable, '-m', 'experiments.posthoc_stage3_s1.scripts.run_s1_development_gpu', '--root', str(repo), '--output', str(output), '--canary-receipt', str(canary_receipt)], cwd=runtime_root, env=env, check=True)
print('Frozen development study complete or safely selection-stopped.')


In [ ]:
subprocess.run([sys.executable, '-m', 'experiments.posthoc_stage3_s1.scripts.package_s1_development_return', '--output', str(output)], cwd=runtime_root, env=env, check=True)


Download `/kaggle/working/s1_development_return_artifacts.zip`. Do not execute confirmation or any S1 final/post-hoc evaluation.